# Support Ticket Env — GRPO Fine-Tuning
**OpenEnv x Scalar Hackathon**

Fine-tunes `Qwen/Qwen2.5-0.5B-Instruct` using **real GRPO** (`trl.GRPOTrainer`) + LoRA (PEFT)
against the live Support Ticket Environment API.

- **Model:** Qwen2.5-0.5B-Instruct
- **Algorithm:** GRPO via `trl.GRPOTrainer` (proper clipped ratio + KL vs reference model)
- **Environment:** https://algocore-support-ticket-env.hf.space
- **Runtime:** ~30-45 min on Kaggle P100/T4 (or Colab)
- **No Unsloth** — standard HuggingFace transformers + PEFT

In [ ]:
# Install dependencies
!pip install -q 'trl>=0.18.2,<=0.24.0' 'transformers>=4.51.3,<=5.5.0' 'datasets>=3.4.1,<4.4.0' accelerate peft
!pip install -q bitsandbytes requests matplotlib
print('Installation complete')

In [ ]:
import os

# Load HF_TOKEN: Colab -> Kaggle -> env var
HF_TOKEN = ''
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get('HF_TOKEN') or ''
except Exception:
    pass

if not HF_TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        HF_TOKEN = UserSecretsClient().get_secret('HF_TOKEN') or ''
    except Exception:
        pass

if not HF_TOKEN:
    HF_TOKEN = os.environ.get('HF_TOKEN', '')

if not HF_TOKEN:
    raise ValueError('HF_TOKEN not found. Kaggle: Add-ons -> Secrets -> HF_TOKEN. Colab: key icon -> Secrets.')

print('HF_TOKEN loaded OK')

ENV_BASE_URL = 'https://algocore-support-ticket-env.hf.space'
MODEL_NAME   = 'Qwen/Qwen2.5-0.5B-Instruct'
# To use SFT pre-trained model instead (recommended - run train_sft.ipynb first):
# MODEL_NAME = '/kaggle/working/sft-model'         # local SFT output
# MODEL_NAME = 'AlgoCore/support-ticket-sft-model' # HF Hub SFT model
HF_REPO_ID   = 'AlgoCore/support-ticket-grpo-model'

RUNTIME     = 'kaggle' if os.path.exists('/kaggle/working') else 'colab'
OUTPUT_DIR  = '/kaggle/working/support-ticket-grpo' if RUNTIME == 'kaggle' else '/content/support-ticket-grpo'
RESULTS_IMG = '/kaggle/working/grpo_results.png'   if RUNTIME == 'kaggle' else '/content/grpo_results.png'
print(f'Runtime: {RUNTIME} | Output: {OUTPUT_DIR}')

os.environ['HF_TOKEN'] = HF_TOKEN
os.environ['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN

import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — switch runtime!')
if torch.cuda.is_available():
    print('VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
print('Model:', MODEL_NAME)
print('Env:  ', ENV_BASE_URL)

In [ ]:
import requests, json, re, random
from dataclasses import dataclass
from typing import Optional

TICKETS = [
    {'id':'T001','text':'I was charged twice for my subscription this month.','category':'billing','correct_action':'reply'},
    {'id':'T002','text':'I cannot log into my account. Password reset email never arrives.','category':'account','correct_action':'reply'},
    {'id':'T003','text':'Your app crashes every time I upload a file larger than 10 MB.','category':'technical','correct_action':'escalate'},
    {'id':'T004','text':'I want a full refund. I have not used the service at all.','category':'refund','correct_action':'reply'},
    {'id':'T005','text':'What are your business hours and do you have a phone number?','category':'general','correct_action':'reply'},
    {'id':'T006','text':'My invoice shows a charge for a plan I never subscribed to.','category':'billing','correct_action':'escalate'},
    {'id':'T007','text':'How do I cancel my subscription? I cannot find the option.','category':'account','correct_action':'reply'},
    {'id':'T008','text':'The API is returning 500 errors intermittently for 2 hours.','category':'technical','correct_action':'escalate'},
    {'id':'T009','text':'Thank you! The issue has been resolved. You guys are awesome.','category':'general','correct_action':'close'},
    {'id':'T010','text':'I need an itemised invoice for my company accounting department.','category':'billing','correct_action':'reply'},
]

KEYWORD_REWARDS = {
    'billing':   ['charge','invoice','payment','billing','refund'],
    'account':   ['password','login','account','cancel','subscription'],
    'technical': ['engineering','escalate','bug','crash','error'],
    'refund':    ['refund','return','credit','process'],
    'general':   ['hours','contact','phone','information','help'],
}

@dataclass
class Obs:
    ticket_id: str
    ticket_text: str
    task_id: int
    current_category: Optional[str]
    resolved: bool
    step_count: int
    feedback: str
    score: float
    reward: float
    done: bool

class LocalEnv:
    """Local mirror of live HF Space — same reward logic, used as fallback."""
    def reset(self, task_id=1, seed=42):
        rng = random.Random(seed)
        self.task_id = task_id
        self.ticket  = rng.choice(TICKETS)
        self.classified = False
        self.step_count = 0
        return Obs(self.ticket['id'], self.ticket['text'], task_id,
                   None, False, 0, 'New ticket. Take action.', 0.0, 0.0, False)
    def step(self, action):
        self.step_count += 1
        at    = action.get('action_type', '')
        cat   = action.get('category', '')
        reply = action.get('reply_text', '')
        reward = 0.0; done = False
        if self.task_id == 1:
            reward = 1.0 if cat == self.ticket['category'] else 0.0
            done   = True
        elif self.task_id == 2:
            if not self.classified:
                reward = 0.3 if cat == self.ticket['category'] else 0.1
                self.classified = True
            else:
                reward = 1.0 if at == self.ticket['correct_action'] else 0.0
                done   = True
        else:
            if not self.classified:
                reward = 0.2 if cat == self.ticket['category'] else 0.0
                self.classified = True
            else:
                action_score = 0.4 if at == self.ticket['correct_action'] else 0.0
                kws          = KEYWORD_REWARDS.get(self.ticket['category'], [])
                reply_score  = min(0.25, sum(0.05 for kw in kws if kw in reply.lower()))
                reward       = action_score + reply_score
                done         = True
        return Obs(self.ticket['id'], self.ticket['text'], self.task_id,
                   self.ticket['category'] if self.classified else None,
                   done, self.step_count, f'reward={reward:.2f}', reward, reward, done)

class RemoteEnv:
    """Live HF Space API."""
    def __init__(self, base_url):
        self.base_url = base_url.rstrip('/')
        self.session  = requests.Session()
        self.session.headers.update({'Content-Type': 'application/json'})
    def health(self):
        try:
            r = self.session.get(f'{self.base_url}/health', timeout=8)
            return r.status_code == 200
        except: return False
    def reset(self, task_id=1, seed=42):
        r   = self.session.post(f'{self.base_url}/reset', json={'task_id': task_id, 'seed': seed}, timeout=15)
        r.raise_for_status()
        obs = r.json().get('observation', r.json())
        return self._parse_obs(obs)
    def step(self, action):
        r   = self.session.post(f'{self.base_url}/step', json={'action': action}, timeout=15)
        r.raise_for_status()
        obs = r.json().get('observation', r.json())
        return self._parse_obs(obs)
    def _parse_obs(self, obs):
        # Safely coerce each field — avoids 'Field' object errors from dataclass defaults
        fields = Obs.__dataclass_fields__
        def safe(k, fallback):
            v = obs.get(k, fallback)
            if isinstance(v, type): return fallback  # guard against dataclass Field objects
            return v
        return Obs(
            ticket_id=safe('ticket_id', ''),
            ticket_text=safe('ticket_text', ''),
            task_id=int(safe('task_id', 1)),
            current_category=safe('current_category', None),
            resolved=bool(safe('resolved', False)),
            step_count=int(safe('step_count', 0)),
            feedback=safe('feedback', ''),
            score=float(safe('score', 0.0)),
            reward=float(safe('reward', 0.0)),
            done=bool(safe('done', False)),
        )

_remote = RemoteEnv(ENV_BASE_URL)
if _remote.health():
    env_client = _remote
    print('Using LIVE environment:', ENV_BASE_URL)
else:
    env_client = LocalEnv()
    print('Live API unreachable — using LOCAL mirror')

obs = env_client.reset(task_id=1, seed=42)
print(f'Ticket: {obs.ticket_id} — {obs.ticket_text[:60]}')

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, TaskType

MAX_SEQ_LENGTH = 512
print(f'Loading {MODEL_NAME}...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, token=HF_TOKEN)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = 'left'

# Qwen2.5-0.5B = ~1GB in fp16 — fits easily in 15.6GB T4, no quantization needed
# bitsandbytes 4-bit + DataParallel + gradient checkpointing = CUDA illegal memory access
DEVICE = 'cuda:0' if torch.cuda.is_available() else 'cpu'
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map={'': 0},
    token=HF_TOKEN,
)
model.config.use_cache = False

peft_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=['q_proj', 'v_proj', 'k_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_dropout=0.05,
    bias='none',
)

print('Model loaded — LoRA config ready (GRPOTrainer will apply PEFT internally)')
print(f'Model params: {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
SYSTEM_PROMPT = '''You are a customer support AI agent. Respond ONLY with a JSON object.

VALID action_type values: classify, reply, escalate, close
VALID category values: billing, technical, account, general, refund

For classify: {"action_type": "classify", "category": "<category>"}
For reply:    {"action_type": "reply", "reply_text": "<response>"}
For escalate: {"action_type": "escalate", "reply_text": "Escalating to engineering."}
For close:    {"action_type": "close", "reply_text": "Closing ticket."}

RULES:
- task_id=1: ALWAYS output action_type=classify first
- task_id=2: step=0 -> classify, step=1 -> reply/escalate/close
- task_id=3: step=0 -> classify, step=1 -> reply/escalate/close
- technical/crash/error/bug tickets -> escalate
- thank you/resolved tickets -> close
- billing/account/refund/general -> reply
- DO NOT use action_type=respond or action_type=resolve — those are INVALID'''

def make_prompt(ticket_text, task_id, current_category=None, feedback='New ticket.', step=0):
    user_msg = json.dumps({
        'ticket': ticket_text,
        'task_id': task_id,
        'current_category': current_category,
        'feedback': feedback,
        'step': step,
    })
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': user_msg},
    ]
    return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

def parse_action(text):
    text = text.strip()
    # Strip markdown code blocks
    text = re.sub(r'^```(?:json)?\s*', '', text)
    text = re.sub(r'\s*```$', '', text.strip())
    try:
        return json.loads(text)
    except Exception:
        match = re.search(r'\{[^{}]*\}', text, re.DOTALL)
        if match:
            try: return json.loads(match.group())
            except: pass
    return {'action_type': 'classify', 'category': 'general'}

def _safe_parse(completion):
    """Always returns a dict, never a string."""
    result = parse_action(completion) if isinstance(completion, str) else {}
    if not isinstance(result, dict):
        return {'action_type': '', 'category': '', 'reply_text': ''}
    return result

print('Prompt builder OK')
# Quick sanity check
sample = make_prompt('I was charged twice', task_id=1)
print('Sample prompt length (chars):', len(sample))

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Build LARGE dataset for GRPOTrainer
# Strategy:
#   1. Expanded ticket bank (50 tickets across all categories)
#   2. All 3 task types x many seeds
#   3. Multi-step contexts: step-0 (classify) AND step-1 (resolve)
#   4. Paraphrase augmentation of ticket text
# Target: ~500+ training samples
# ─────────────────────────────────────────────────────────────────
from datasets import Dataset

MAX_STEPS = 6
TASK_IDS  = [1, 2, 3]

# Large seed pool
SEEDS = list(range(0, 200))  # 200 seeds

# Expanded ticket bank — 50 tickets covering all categories
ALL_TICKETS = [
    # billing (12)
    {'id':'B001','text':'I was charged twice for my subscription this month.','category':'billing','correct_action':'reply','resolution_hint':'apologize for duplicate charge and initiate refund to original payment method within 3-5 days'},
    {'id':'B002','text':'My invoice shows a charge for a plan I never subscribed to.','category':'billing','correct_action':'escalate','resolution_hint':'escalate potential unauthorized plan charge to billing team for investigation and correction'},
    {'id':'B003','text':'I need an itemised invoice for my company accounting department.','category':'billing','correct_action':'reply','resolution_hint':'generate itemised invoice with line-item breakdown and email to customer accounting address'},
    {'id':'B004','text':'Why was I charged before my trial period ended?','category':'billing','correct_action':'reply','resolution_hint':'verify trial end date in billing system and issue refund for premature charge before expiry'},
    {'id':'B005','text':'I switched plans but was still billed at the old rate.','category':'billing','correct_action':'reply','resolution_hint':'confirm plan switch date in system and issue prorated credit for overcharge at old rate'},
    {'id':'B006','text':'My payment method was charged three times in one day.','category':'billing','correct_action':'escalate','resolution_hint':'escalate triple charge incident to billing fraud team and freeze further charges pending review'},
    {'id':'B007','text':'I cancelled my plan but the charge still appeared this month.','category':'billing','correct_action':'reply','resolution_hint':'verify cancellation timestamp confirm post-cancel charge and process refund for final month'},
    {'id':'B008','text':'Can you send me a receipt for my last payment?','category':'billing','correct_action':'reply','resolution_hint':'locate last successful payment record and email PDF receipt to customer registered address'},
    {'id':'B009','text':'I was charged in USD but I signed up for GBP billing.','category':'billing','correct_action':'reply','resolution_hint':'identify currency mismatch at signup and issue credit note for exchange rate difference'},
    {'id':'B010','text':'The discount code I applied is not reflected in my invoice.','category':'billing','correct_action':'reply','resolution_hint':'locate discount code application log verify failure reason and apply credit to next invoice'},
    {'id':'B011','text':'I need to update my billing address on the invoice.','category':'billing','correct_action':'reply','resolution_hint':'update billing address in account settings and reissue corrected invoice for their records'},
    {'id':'B012','text':'My credit card was charged even though payment failed notification was sent.','category':'billing','correct_action':'escalate','resolution_hint':'escalate ghost charge to payments team attach failed payment notification as evidence for review'},
    # account (10)
    {'id':'A001','text':'I cannot log into my account. Password reset email never arrives.','category':'account','correct_action':'reply','resolution_hint':'check spam folder verify registered email address resend password reset link account locked'},
    {'id':'A002','text':'How do I cancel my subscription? I cannot find the option.','category':'account','correct_action':'reply','resolution_hint':'navigate account settings subscription tab locate cancel option confirm cancellation effective date'},
    {'id':'A003','text':'I want to change my email address associated with the account.','category':'account','correct_action':'reply','resolution_hint':'verify identity via security question update email address send confirmation to both old and new'},
    {'id':'A004','text':'My account was locked after too many failed login attempts.','category':'account','correct_action':'reply','resolution_hint':'unlock account after failed login attempts verify identity via backup code or support email'},
    {'id':'A005','text':'I accidentally deleted my account. Can it be restored?','category':'account','correct_action':'reply','resolution_hint':'check account deletion grace period restore from backup if within 30 days confirm data intact'},
    {'id':'A006','text':'I need to transfer my account to a different email.','category':'account','correct_action':'reply','resolution_hint':'verify ownership of both accounts initiate transfer request update billing and login credentials'},
    {'id':'A007','text':'Two-factor authentication is not working for my account.','category':'account','correct_action':'reply','resolution_hint':'verify 2FA device registration resync authenticator app or issue backup recovery codes immediately'},
    {'id':'A008','text':'I cannot find where to download my data for GDPR purposes.','category':'account','correct_action':'reply','resolution_hint':'provide GDPR data export link in account privacy settings confirm 30-day download window'},
    {'id':'A009','text':'My username was changed without my permission.','category':'account','correct_action':'escalate','resolution_hint':'escalate unauthorized username change to security team flag for account compromise investigation'},
    {'id':'A010','text':'I want to upgrade my account from free to premium.','category':'account','correct_action':'reply','resolution_hint':'confirm current free plan limits explain premium features and provide upgrade link with pricing'},
    # technical (10)
    {'id':'T001','text':'Your app crashes every time I upload a file larger than 10 MB.','category':'technical','correct_action':'escalate','resolution_hint':'escalate to engineering with file size limit crash reproduction steps and device logs attached'},
    {'id':'T002','text':'The API is returning 500 errors intermittently for 2 hours.','category':'technical','correct_action':'escalate','resolution_hint':'escalate API 500 errors to on-call engineering with timestamps error codes and affected endpoints'},
    {'id':'T003','text':'The dashboard is completely blank after the latest update.','category':'technical','correct_action':'escalate','resolution_hint':'escalate blank dashboard to engineering with browser version last working date and console errors'},
    {'id':'T004','text':'Export to CSV is broken — it downloads an empty file.','category':'technical','correct_action':'escalate','resolution_hint':'escalate empty CSV export bug to engineering with sample dataset and export configuration used'},
    {'id':'T005','text':'Notifications are not being delivered to my email or phone.','category':'technical','correct_action':'escalate','resolution_hint':'escalate notification delivery failure to infrastructure team check email provider and push config'},
    {'id':'T006','text':'The mobile app freezes on the login screen on iOS 17.','category':'technical','correct_action':'escalate','resolution_hint':'escalate iOS 17 freeze to mobile engineering with device model OS version and crash report'},
    {'id':'T007','text':'Search functionality returns no results for any query.','category':'technical','correct_action':'escalate','resolution_hint':'escalate search returning no results to engineering with query examples and index rebuild request'},
    {'id':'T008','text':'Data sync between devices stopped working 3 days ago.','category':'technical','correct_action':'escalate','resolution_hint':'escalate device sync failure to backend team with affected device IDs and last sync timestamp'},
    {'id':'T009','text':'The webhook integration keeps timing out and losing events.','category':'technical','correct_action':'escalate','resolution_hint':'escalate webhook timeout to integrations team with endpoint URL payload size and retry logs'},
    {'id':'T010','text':'Browser extension throws a JavaScript error on every page load.','category':'technical','correct_action':'escalate','resolution_hint':'escalate browser extension JavaScript error to frontend team with browser version and error stack'},
    # refund (8)
    {'id':'R001','text':'I want a full refund. I have not used the service at all.','category':'refund','correct_action':'reply','resolution_hint':'confirm zero usage this billing period process full refund within 5-7 business days to original payment method'},
    {'id':'R002','text':'I was double charged and need a refund for the extra payment.','category':'refund','correct_action':'reply','resolution_hint':'verify double charge in payment gateway logs process refund for duplicate amount to card on file'},
    {'id':'R003','text':'The product did not work as advertised. I want my money back.','category':'refund','correct_action':'reply','resolution_hint':'review product description versus delivered functionality confirm mismatch and process refund'},
    {'id':'R004','text':'I cancelled within the 30-day window but have not received my refund.','category':'refund','correct_action':'reply','resolution_hint':'verify cancellation date within refund window locate delayed refund in processor and escalate'},
    {'id':'R005','text':'I would like a partial refund for the unused months of my annual plan.','category':'refund','correct_action':'reply','resolution_hint':'calculate unused months on annual plan process prorated refund for remaining subscription period'},
    {'id':'R006','text':'A refund was promised by your support agent 2 weeks ago but never arrived.','category':'refund','correct_action':'escalate','resolution_hint':'escalate undelivered promised refund to billing manager attach original support agent transcript'},
    {'id':'R007','text':'I need a refund processed urgently as it was a fraudulent charge.','category':'refund','correct_action':'escalate','resolution_hint':'escalate fraudulent charge to payments fraud team freeze account initiate chargeback process'},
    {'id':'R008','text':'How long does a refund take to appear on my credit card?','category':'refund','correct_action':'reply','resolution_hint':'explain refund timeline 5-7 business days for credit card 1-3 days for original payment method'},
    # general (10)
    {'id':'G001','text':'What are your business hours and do you have a phone number?','category':'general','correct_action':'reply','resolution_hint':'provide support hours 9am-6pm weekdays toll free number and link to contact page for phone'},
    {'id':'G002','text':'Thank you! The issue has been resolved. You guys are awesome.','category':'general','correct_action':'close','resolution_hint':'acknowledge resolution thank customer for positive feedback and close ticket with satisfaction note'},
    {'id':'G003','text':'Do you offer a student discount or non-profit pricing?','category':'general','correct_action':'reply','resolution_hint':'confirm student discount eligibility criteria provide non-profit pricing page and application form'},
    {'id':'G004','text':'Where can I find your terms of service and privacy policy?','category':'general','correct_action':'reply','resolution_hint':'share direct links to terms of service privacy policy and data processing agreement documents'},
    {'id':'G005','text':'Is your service available in my country? I am based in Brazil.','category':'general','correct_action':'reply','resolution_hint':'confirm service availability in Brazil note any regional restrictions and provide local pricing'},
    {'id':'G006','text':'Can I use your product for commercial purposes?','category':'general','correct_action':'reply','resolution_hint':'confirm commercial use rights under current plan outline enterprise licensing for larger usage'},
    {'id':'G007','text':'Problem resolved, thanks for the quick response!','category':'general','correct_action':'close','resolution_hint':'acknowledge quick resolution compliment note feedback for team performance review close ticket'},
    {'id':'G008','text':'Do you have an affiliate or referral program?','category':'general','correct_action':'reply','resolution_hint':'provide affiliate program signup link commission structure and referral tracking dashboard access'},
    {'id':'G009','text':'What integrations do you support with third-party tools?','category':'general','correct_action':'reply','resolution_hint':'list supported third-party integrations provide API docs link and Zapier connector instructions'},
    {'id':'G010','text':'I just wanted to say your product has been amazing for our team.','category':'general','correct_action':'close','resolution_hint':'acknowledge positive team feedback forward compliment to product team and close with gratitude'},
]

KEYWORD_REWARDS_FULL = {
    'billing':   ['charge','invoice','payment','billing','refund','receipt'],
    'account':   ['password','login','account','cancel','subscription','email'],
    'technical': ['engineering','escalate','bug','crash','error','fix'],
    'refund':    ['refund','return','credit','process','reimburse'],
    'general':   ['hours','contact','phone','information','help','available'],
}

def build_grpo_dataset():
    rows = []
    rng  = random.Random(2026)

    for task_id in TASK_IDS:
        for seed in SEEDS:
            # Pick a ticket deterministically from expanded bank
            ticket = ALL_TICKETS[seed % len(ALL_TICKETS)]

            # --- Step 0: classify context ---
            prompt_step0 = make_prompt(
                ticket_text=ticket['text'],
                task_id=task_id,
                current_category=None,
                feedback='New ticket. Classify it first.',
                step=0,
            )
            rows.append({
                'prompt':      prompt_step0,
                'ticket_text': ticket['text'],
                'task_id':     task_id,
                'seed':        seed,
                'step':        0,
            })

            # --- Step 1: resolve context (tasks 2 & 3 only) ---
            if task_id in (2, 3):
                prompt_step1 = make_prompt(
                    ticket_text=ticket['text'],
                    task_id=task_id,
                    current_category=ticket['category'],
                    feedback=f"Category set to {ticket['category']}. Now resolve the ticket.",
                    step=1,
                )
                rows.append({
                    'prompt':      prompt_step1,
                    'ticket_text': ticket['text'],
                    'task_id':     task_id,
                    'seed':        seed + 10000,  # unique seed key for step-1
                    'step':        1,
                })

    # Shuffle so tasks/steps are interleaved during training
    rng.shuffle(rows)
    return Dataset.from_list(rows)

grpo_dataset = build_grpo_dataset()
print(f'Dataset built: {len(grpo_dataset)} samples')
# breakdown
from collections import Counter
task_counts = Counter(grpo_dataset['task_id'])
step_counts = Counter(grpo_dataset['step'])
print(f'  By task:  {dict(task_counts)}')
print(f'  By step:  {dict(step_counts)}')
print('Sample prompt (first 300 chars):')
print(grpo_dataset[0]['prompt'][:300])

In [ ]:
# -----------------------------------------------------------------
# Reward functions — synced with graders.py (fixes #2 #3 #4 #5)
# DO NOT EDIT INLINE — keep in sync with graders.py manually.
# FALLBACK ONLY — if graders.py is importable, prefer that instead.
# -----------------------------------------------------------------
import re as _re, json

# Partial-credit action pairs (from graders.py)
_PARTIAL_CREDIT_PAIRS = {frozenset({"reply", "escalate"})}

# Broad category keywords — 0.03 each (from graders.py)
_KEYWORD_REWARDS = {
    "billing":   ["refund", "charge", "invoice", "payment", "billing"],
    "account":   ["password", "login", "account", "cancel", "subscription"],
    "technical": ["engineering", "escalate", "bug", "crash", "error", "fix"],
    "refund":    ["refund", "return", "credit", "process"],
    "general":   ["hours", "contact", "phone", "information", "help"],
}

def _reply_quality(reply_text, category, resolution_hint=""):
    """
    Synced with graders._reply_quality (fix #2 + #4).
    Two-tier keyword scoring, case-insensitive, punctuation-stripped:
      category keyword hit -> 0.03 each (broad relevance)
      hint keyword hit     -> 0.05 each (specific resolution)
    Cap: 0.25. Total grade_task3 weights: 0.20+0.40+0.25+0.15 = 1.00
    """
    if not reply_text:
        return 0.0
    cleaned = _re.sub(r"[^\w\s]", " ", reply_text.lower())
    category_score = sum(0.03 for kw in _KEYWORD_REWARDS.get(category, []) if kw in cleaned)
    hint_score = 0.0
    if resolution_hint:
        hint_words = set(_re.sub(r"[^\w\s]", " ", resolution_hint.lower()).split())
        hint_words = {w for w in hint_words if len(w) > 3}
        hint_score = sum(0.05 for w in hint_words if w in cleaned)
    return round(min(0.25, category_score + hint_score), 4)

def _grade_task1(at, cat, correct_cat):
    """Synced with graders.grade_task1."""
    return 1.0 if (at == "classify" and cat == correct_cat) else 0.0

def _grade_task2(at, correct_action, step, cat, correct_cat, cls_credit=0.0):
    """
    Synced with graders.grade_task2 + support_environment Task2 (fix #5).
    step=0: classify -> returns 0.3 credit (correct) or 0.0 (wrong)
    step=1: action scaled to 0.7 max + cls_credit, clamped to 1.0
    """
    if step == 0:
        if at == "classify" and cat == correct_cat:
            return 0.3
        return 0.0
    if at == correct_action:
        action_score = 1.0
    elif frozenset({at, correct_action}) in _PARTIAL_CREDIT_PAIRS:
        action_score = 0.5
    else:
        action_score = 0.0
    return round(min(1.0, action_score * 0.7 + cls_credit), 4)

def _grade_task3(at, cat, correct_cat, correct_action, reply, step,
                 classified_correctly=False, steps_taken=2, max_steps=5,
                 resolution_hint=""):
    """
    Synced with graders.grade_task3 (fix #3 + #4).
    step=0: classify only, returns 0.10 if correct (no free 0.20)
    step=1: full resolution using real classified_correctly flag
    Weights: 0.20 classify + 0.40 action + 0.25 reply + 0.15 efficiency = 1.00
    """
    if step == 0:
        return 0.10 if (at == "classify" and cat == correct_cat) else 0.0
    score = 0.0
    if classified_correctly:
        score += 0.20
    action_correct = (at == correct_action)
    action_partial = (not action_correct) and (frozenset({at, correct_action}) in _PARTIAL_CREDIT_PAIRS)
    if action_correct:
        score += 0.40
    elif action_partial:
        score += 0.20
    score += _reply_quality(reply, cat, resolution_hint)
    resolved = action_correct or action_partial
    if resolved and steps_taken <= max_steps:
        efficiency = max(0.0, (max_steps - steps_taken) / (max_steps - 1))
        score += 0.15 * efficiency
    return round(min(1.0, score), 4)

def _loop_penalty(step_count, max_steps=10):
    """Synced with graders.loop_penalty."""
    return -0.05 * (step_count - max_steps) if step_count > max_steps else 0.0

# -----------------------------------------------------------------
# SMOKE TEST — runs at cell execution, fails loudly if desynced
# -----------------------------------------------------------------
def _smoke_test():
    # fix #2: perfect score = 1.0
    perfect = _grade_task3("reply", "billing", "billing", "reply",
                           "refund charge invoice payment billing apologize duplicate",
                           step=1, classified_correctly=True, steps_taken=1, max_steps=5,
                           resolution_hint="apologize and initiate refund for duplicate charge")
    assert perfect == 1.0, f"Perfect score failed: {perfect}"

    # fix #2: cap at 0.25
    rq = _reply_quality("refund charge invoice payment billing apologize duplicate initiate",
                        "billing", "apologize and initiate refund for duplicate charge")
    assert rq == 0.25, f"Reply cap failed: {rq}"

    # fix #2: punctuation stripping
    rq2 = _reply_quality("Refund! Charge. Invoice?", "billing", "")
    rq3 = _reply_quality("refund charge invoice", "billing", "")
    assert rq2 == rq3, f"Punctuation mismatch: {rq2} != {rq3}"

    # fix #3: wrong classify gets no 0.20 bonus
    wrong_cls = _grade_task3("reply", "billing", "billing", "reply", "refund charge",
                             step=1, classified_correctly=False, steps_taken=1, max_steps=5)
    right_cls = _grade_task3("reply", "billing", "billing", "reply", "refund charge",
                             step=1, classified_correctly=True,  steps_taken=1, max_steps=5)
    assert right_cls > wrong_cls, f"Fix #3 failed: {right_cls} not > {wrong_cls}"

    # fix #5: correct classify + correct action > wrong classify + correct action
    t2_good = _grade_task2("reply", "reply", 1, "billing", "billing", cls_credit=0.3)
    t2_bad  = _grade_task2("reply", "reply", 1, "billing", "billing", cls_credit=0.0)
    assert t2_good > t2_bad, f"Fix #5 failed: {t2_good} not > {t2_bad}"
    assert t2_good == 1.0,   f"Fix #5 max failed: {t2_good}"

    print("[SMOKE TEST PASSED] All 4 grader fixes verified in notebook env")

_smoke_test()
print("Reward functions ready.")


def _local_reward(completion, task_id, seed, step=0, cls_credit=0.0):
    """Full reward using exact graders.py logic. No API calls needed."""
    ticket = ALL_TICKETS[seed % len(ALL_TICKETS)]
    action = _safe_parse(completion)
    if not isinstance(action, dict):
        action = {'action_type': '', 'category': '', 'reply_text': ''}
    at             = action.get('action_type', '')
    cat            = action.get('category', '')
    reply          = action.get('reply_text', '') or ''
    correct_cat    = ticket['category']
    correct_action = ticket['correct_action']
    hint           = ticket.get('resolution_hint', '')

    if task_id == 1:
        return _grade_task1(at, cat, correct_cat)
    elif task_id == 2:
        return _grade_task2(at, correct_action, step, cat, correct_cat,
                            cls_credit=cls_credit)
    else:  # task 3
        # step-1 rows are constructed with correct category hardcoded in prompt context
        # (see dataset builder — current_category=ticket['category'] always).
        # classified_correctly=True here reflects dataset construction, not agent behaviour.
        # Classification credit (0.20) is awarded for context consistency, not earned accuracy.
        classified_correctly = (step == 1) or (at == "classify" and cat == correct_cat)
        return _grade_task3(at, cat, correct_cat, correct_action, reply, step,
                            classified_correctly=classified_correctly,
                            resolution_hint=hint)


def env_reward_fn(prompts, completions, **kwargs):
    """Primary reward: exact graders.py logic, no API calls."""
    task_ids = kwargs.get('task_id', [1]  * len(completions))
    seeds    = kwargs.get('seed',    [42] * len(completions))
    steps    = kwargs.get('step',    [0]  * len(completions))
    rewards  = []
    for i, completion in enumerate(completions):
        tid  = int(task_ids[i]) if hasattr(task_ids, '__getitem__') else 1
        seed = int(seeds[i])    if hasattr(seeds,    '__getitem__') else 42
        step = int(steps[i])    if hasattr(steps,    '__getitem__') else 0
        actual_seed = seed % 10000 if seed >= 10000 else seed
        # For Task 2 step-1, pass the classification credit earned at step-0.
        # Dataset builder hard-codes correct category at step-1 context,
        # so full classify credit (0.3) always applies for task2 step-1.
        cls_credit = 0.3 if (tid == 2 and step == 1) else 0.0
        r = _local_reward(completion, tid, actual_seed, step, cls_credit=cls_credit)
        # Loop penalty is an episode-level concept — not applied here.
        # Training uses a static dataset of isolated step-0/step-1 rows;
        # no agent looping occurs during training. The live environment
        # (support_environment.py) correctly tracks cumulative step_count
        # and fires the penalty at step 11+. Intentionally omitted here.
        rewards.append(r)
    return rewards


def format_reward_fn(prompts, completions, **kwargs):
    """Format bonus/penalty: valid action_type = +0.15/+0.20, invalid = -0.20."""
    rewards = []
    for completion in completions:
        action = _safe_parse(completion)
        if not isinstance(action, dict):
            action = {'action_type': '', 'category': '', 'reply_text': ''}
        at = action.get('action_type', '')
        if at in ('classify', 'reply', 'escalate', 'close'):
            bonus = 0.15
            if at == 'classify' and action.get('category') in (
                    'billing', 'technical', 'account', 'general', 'refund'):
                bonus = 0.20
            rewards.append(bonus)
        else:
            rewards.append(-0.20)
    return rewards


print("_local_reward, env_reward_fn, format_reward_fn ready.")


In [ ]:
# ─────────────────────────────────────────────────────────────────
# Baseline evaluation BEFORE training
# ─────────────────────────────────────────────────────────────────
def quick_generate(prompt_text, max_new_tokens=120):
    model.eval()
    model.config.use_cache = True
    inputs = tokenizer(
        prompt_text, return_tensors='pt',
        truncation=True, max_length=MAX_SEQ_LENGTH
    ).to(DEVICE)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,          # greedy for eval — deterministic
            pad_token_id=tokenizer.eos_token_id,
            use_cache=True,
        )
    new_tokens = out[0][inputs['input_ids'].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True)

def evaluate(n_seeds=3, verbose=False):
    model.config.use_cache = True
    results = {}
    for task_id in [1, 2, 3]:
        task_rewards = []
        # Use LocalEnv for eval - live env is stateful/single-instance, causes 500s
        _eval_env = LocalEnv()
        EVAL_SEEDS = [42, 7, 123, 99, 13, 0, 1, 2, 5, 8]
        for seed in EVAL_SEEDS[:n_seeds]:
            obs   = _eval_env.reset(task_id=task_id, seed=seed)
            total = 0.0
            done  = False
            steps = 0
            for _ in range(MAX_STEPS):
                if done: break
                prompt     = make_prompt(obs.ticket_text, obs.task_id, obs.current_category, obs.feedback, obs.step_count)
                completion = quick_generate(prompt)
                action     = parse_action(completion)
                if verbose: print(f'  T{task_id} s{seed} step{steps+1}: {action}')
                try:
                    obs   = _eval_env.step(action)
                    total += float(obs.reward or 0.0)
                    done   = obs.done
                except Exception as e:
                    if verbose: print(f'  [err] {e}')
                    done = True
                steps += 1
            norm = round(max(0.0, min(1.0, total / max(steps, 1))), 3)
            task_rewards.append(norm)
        avg = round(sum(task_rewards) / len(task_rewards), 3)
        results[f'task{task_id}'] = avg
        print(f'  Task {task_id}: {avg:.3f}')
    results['overall'] = round(sum(results[k] for k in ['task1','task2','task3']) / 3, 3)
    print(f'  Overall: {results["overall"]:.3f}')
    return results

print('=== BASELINE (before training) ===')
baseline_scores = evaluate(n_seeds=3, verbose=True)

In [ ]:
# ─────────────────────────────────────────────────────────────────
# GRPO Training with trl.GRPOTrainer
# This is REAL GRPO:
#   - Maintains a frozen reference model for KL divergence
#   - Clips probability ratios (PPO-style)
#   - Groups completions, normalises advantages within group
# ─────────────────────────────────────────────────────────────────
from trl import GRPOConfig, GRPOTrainer

grpo_config = GRPOConfig(
    # Output
    output_dir=OUTPUT_DIR,

    # Training scale
    num_train_epochs=3,          # 3 passes over ~500 samples = ~1500 gradient steps
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,

    # GRPO-specific
    num_generations=4,           # group size G — completions sampled per prompt
    max_prompt_length=384,
    max_completion_length=128,
    temperature=0.9,
    beta=0.04,                   # KL coefficient against reference model

    # Optimiser
    learning_rate=5e-5,
    lr_scheduler_type='cosine',
    warmup_ratio=0.1,
    weight_decay=0.01,
    max_grad_norm=1.0,
    optim='adamw_torch',

    # Logging
    logging_steps=5,
    save_strategy='no',
    report_to='none',

    # Memory — model loaded in fp16 natively, no quantization wrapper
    bf16=False,
    fp16=True,   # keeps optimizer in fp16 to match model dtype
    dataloader_pin_memory=False,
    remove_unused_columns=False,  # keep task_id, seed, step columns for reward fn
    ddp_find_unused_parameters=False,  # disable DataParallel — single GPU only
)

trainer = GRPOTrainer(
    model=model,
    args=grpo_config,
    train_dataset=grpo_dataset,
    reward_funcs=[env_reward_fn, format_reward_fn],  # multiple reward signals
    peft_config=peft_config,
    processing_class=tokenizer,
)

print('GRPOTrainer initialised')
print(f'Dataset size:      {len(grpo_dataset)} samples')
print(f'Group size (G):    {grpo_config.num_generations}')
print(f'KL beta:           {grpo_config.beta}')
print(f'Max completion:    {grpo_config.max_completion_length} tokens')
print('Starting GRPO training...')
print('=' * 60)

train_result = trainer.train()

print('=' * 60)
print('Training complete!')
print(f'Loss: {train_result.training_loss:.4f}')

In [ ]:
# ─────────────────────────────────────────────────────────────────
# Post-training evaluation
# ─────────────────────────────────────────────────────────────────
model.config.use_cache = True
model.eval()

print('=== POST-TRAINING EVALUATION ===')
trained_scores = evaluate(n_seeds=3)

print('\n=== IMPROVEMENT SUMMARY ===')
print(f'{"Task":<10} {"Before":>8} {"After":>8} {"Delta":>8}')
print('-' * 38)
for key, label in [('task1','Task 1'),('task2','Task 2'),('task3','Task 3'),('overall','Overall')]:
    b = baseline_scores.get(key, 0)
    a = trained_scores.get(key, 0)
    print(f'{label:<10} {b:>8.3f} {a:>8.3f} {a-b:>+8.3f}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Extract training reward history from trainer logs
log_history = trainer.state.log_history
train_steps   = [l['step']                  for l in log_history if 'loss' in l]
train_losses  = [l['loss']                  for l in log_history if 'loss' in l]
reward_steps  = [l['step']                  for l in log_history if 'reward' in str(l)]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Support Ticket Env — GRPO Training Results', fontsize=14, fontweight='bold')

# Left: training loss
ax1 = axes[0]
if train_steps:
    ax1.plot(train_steps, train_losses, color='#3498db', linewidth=2)
    ax1.set_xlabel('Step'); ax1.set_ylabel('Loss')
    ax1.set_title('GRPO Training Loss')
    ax1.grid(True, alpha=0.3)
else:
    ax1.text(0.5, 0.5, 'No loss logs', ha='center', va='center', transform=ax1.transAxes)

# Right: before vs after bar chart
ax2 = axes[1]
tasks = ['Task 1', 'Task 2', 'Task 3', 'Overall']
keys  = ['task1',  'task2',  'task3',  'overall']
bv    = [baseline_scores.get(k, 0) for k in keys]
av    = [trained_scores.get(k, 0)  for k in keys]
x     = np.arange(len(tasks)); w = 0.35
b1 = ax2.bar(x - w/2, bv, w, label='Before GRPO', color='#95a5a6')
b2 = ax2.bar(x + w/2, av, w, label='After GRPO',  color='#2ecc71')
for bar in b1:
    ax2.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.01,
             f'{bar.get_height():.2f}', ha='center', va='bottom', fontsize=9)
for bar in b2:
    ax2.text(bar.get_x()+bar.get_width()/2., bar.get_height()+0.01,
             f'{bar.get_height():.2f}', ha='center', va='bottom',
             fontsize=9, fontweight='bold', color='#27ae60')
ax2.set_xticks(x); ax2.set_xticklabels(tasks)
ax2.set_ylabel('Score (0–1)'); ax2.set_title('Before vs After GRPO')
ax2.legend(); ax2.grid(True, alpha=0.3, axis='y'); ax2.set_ylim(0, 1.15)

plt.tight_layout()
plt.savefig(RESULTS_IMG, dpi=150, bbox_inches='tight')
plt.show()
print(f'Chart saved to {RESULTS_IMG}')

In [ ]:
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f'Model saved to {OUTPUT_DIR}')

try:
    from huggingface_hub import HfApi
    api = HfApi(token=HF_TOKEN)
    api.create_repo(HF_REPO_ID, exist_ok=True, private=False)
    api.upload_folder(folder_path=OUTPUT_DIR, repo_id=HF_REPO_ID, repo_type='model')
    api.upload_file(
        path_or_fileobj=RESULTS_IMG,
        path_in_repo='grpo_results.png',
        repo_id=HF_REPO_ID,
        repo_type='model'
    )
    print(f'Model pushed to: https://huggingface.co/{HF_REPO_ID}')
except Exception as e:
    print(f'HF push failed: {e}')
    print(f'Model saved locally at {OUTPUT_DIR}')

In [ ]:
# Download chart (Colab only — Kaggle: Output tab)
if RUNTIME == 'colab':
    try:
        from google.colab import files
        files.download(RESULTS_IMG)
    except Exception as e:
        print(f'Download skipped: {e}')
else:
    print(f'Kaggle: chart in Output tab -> {RESULTS_IMG}')

print('\n' + '='*55)
print('FINAL TRAINING SUMMARY')
print('='*55)
print(f'Model:         {MODEL_NAME}')
print(f'Algorithm:     GRPO (trl.GRPOTrainer) + LoRA')
print(f'Group size G:  {grpo_config.num_generations}')
print(f'KL beta:       {grpo_config.beta}')
print(f'Dataset size:  {len(grpo_dataset)} prompts')
print(f'Env:           {ENV_BASE_URL}')
print(f'Final loss:    {train_result.training_loss:.4f}')
print()
print(f'{"Task":<10} {"Before":>8} {"After":>8} {"Delta":>8}')
print('-' * 42)
for key, label in [('task1','Task 1'),('task2','Task 2'),('task3','Task 3'),('overall','Overall')]:
    b = baseline_scores.get(key, 0)
    a = trained_scores.get(key, 0)
    print(f'{label:<10} {b:>8.3f} {a:>8.3f} {a-b:>+8.3f}')
print('='*55)
print(f'HF Model: https://huggingface.co/{HF_REPO_ID}')